In [2]:
import sys
print(f"Python version: {sys.version}")
print(f"Platform: {sys.platform}")

Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: linux


In [6]:
!pip install pyBKT

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 26.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyBKT: filename=pyBKT-1.4.2-cp312-cp312-linux_x86_64.whl size=1130545 sha256=44761a9e85f3a43029f3cd5304311ac2da63613b4f76852697e676f68fdff4d0
  Stored in directory: /root/.cache/pip/wheels/86/91/2f/0eb4389642bed5183ec3d8d025ef7ff02099f08dafb9e0cc19
Successfully built pyBKT


In [3]:
from google.colab import files
uploaded = files.upload()

Saving bkt_training_data.csv to bkt_training_data.csv


In [4]:
import pandas as pd

df = pd.read_csv('bkt_training_data.csv')

print(f"Loaded {len(df):,} interactions")
print(f"Students: {df['user_id'].nunique():,}")
print(f"Skills:   {df['skill_name'].nunique()}")
print(f"Correct rate: {df['correct'].mean():.2%}")
print()
df.head()

Loaded 445,009 interactions
Students: 3,669
Skills:   95
Correct rate: 69.57%



,user_id,skill_name,correct,order_id
0,14,Circle Graph,0,21617623
1,14,Percent Of,0,21617623
2,14,Circle Graph,1,21617632
3,14,Percent Of,1,21617632
4,14,Circle Graph,0,21617641


In [7]:
from pyBKT.models import Model

# Just the 3 most common skills
test_skills = df['skill_name'].value_counts().head(3).index.tolist()
test_df = df[df['skill_name'].isin(test_skills)]

print(f"Testing on {len(test_df):,} interactions across {len(test_skills)} skills")

test_model = Model(seed=42, num_fits=5)
test_model.fit(data=test_df)

print("\nLearned parameters:")
print(test_model.params())

Testing on 69,903 interactions across 3 skills

Learned parameters:
                                                      value
skill                               param   class          
Percent Of                          prior   default 0.05851
                                    learns  default 0.01300
                                    guesses default 0.53396
                                    slips   default 0.00069
                                    forgets default 0.00000
Equation Solving Two or Fewer Steps prior   default 0.61139
                                    learns  default 0.02836
                                    guesses default 0.35532
                                    slips   default 0.22574
                                    forgets default 0.00000
Addition and Subtraction Integers   prior   default 0.54656
                                    learns  default 0.00592
                                    guesses default 0.43869
                                

In [8]:
# Check if convergence was reached
from pyBKT.models import Model

# Re-train one skill with explicit settings and verbose output
single_skill_df = df[df['skill_name'] == 'Percent Of']
print(f"Training on {len(single_skill_df):,} interactions for one skill")

import time
start = time.time()
m = Model(seed=42, num_fits=5)
m.fit(data=single_skill_df)
elapsed = time.time() - start

print(f"\nTime: {elapsed:.2f} seconds")
print(f"\nParameters:")
print(m.params())

# Sanity check: predict on the training data and compute AUC
preds = m.predict(data=single_skill_df)
print(f"\nPrediction columns: {preds.columns.tolist()}")
print(preds.head())

Training on 22,862 interactions for one skill

Time: 0.24 seconds

Parameters:
                             value
skill      param   class          
Percent Of prior   default 0.05851
           learns  default 0.01300
           guesses default 0.53396
           slips   default 0.00069
           forgets default 0.00000

Prediction columns: ['user_id', 'skill_name', 'correct', 'order_id', 'correct_predictions', 'state_predictions']
   user_id  skill_name  correct  order_id  correct_predictions  \
1       14  Percent Of        0  21617623              0.56119   
3       14  Percent Of        1  21617632              0.54005   
5       14  Percent Of        0  21617641              0.55113   
7       14  Percent Of        0  21617650              0.54003   
9       14  Percent Of        0  21617659              0.54002   

   state_predictions  
1            0.05851  
3            0.01309  
5            0.03690  
7            0.01305  
9            0.01302  


In [9]:
import time

print("Training BKT on full dataset...")
print(f"  {len(df):,} interactions, {df['skill_name'].nunique()} skills")
print()

start = time.time()
model = Model(seed=42, num_fits=5)
model.fit(data=df)
elapsed = time.time() - start

print(f"Training complete in {elapsed/60:.1f} minutes.")
print(f"Learned parameters for {df['skill_name'].nunique()} skills.")

Training BKT on full dataset...
  445,009 interactions, 95 skills

Training complete in 0.1 minutes.
Learned parameters for 95 skills.


In [10]:
import pickle

# Save the trained model
with open('bkt_model.pkl', 'wb') as f:
    pickle.dump(model, f)

print(f"Model saved to bkt_model.pkl")

# Look at all the parameters
params = model.params()
print(f"\nLearned parameters for all {df['skill_name'].nunique()} skills:")
print(params)

Model saved to bkt_model.pkl

Learned parameters for all 95 skills:
                                              value
skill                       param   class          
Percent Of                  prior   default 0.05851
                            learns  default 0.01300
                            guesses default 0.53396
                            slips   default 0.00069
                            forgets default 0.00000
...                                             ...
Recognize Quadratic Pattern prior   default 0.49998
                            learns  default 0.02002
                            guesses default 0.00004
                            slips   default 0.04848
                            forgets default 0.00000

[475 rows x 1 columns]


In [11]:
# Reshape into a clean table: one row per skill, one column per parameter
params_table = params.reset_index().pivot(index='skill', columns='param', values='value')
params_table = params_table[['prior', 'learns', 'guesses', 'slips', 'forgets']]

print(f"Total skills modelled: {len(params_table)}")
print()

# Sort by prior (so we can see which skills students come in already knowing)
print("Skills students already know best (highest prior):")
print(params_table.sort_values('prior', ascending=False).head(10))
print()

print("Skills with highest guess rate (multiple-choice-like):")
print(params_table.sort_values('guesses', ascending=False).head(10))
print()

print("Skills students slip on most (high slip rate):")
print(params_table.sort_values('slips', ascending=False).head(10))

Total skills modelled: 95

Skills students already know best (highest prior):
param                                  prior  learns  guesses   slips  forgets
skill                                                                         
Nets of 3D Figures                   0.97876 0.10960  0.46474 0.03278  0.00000
Fraction Of                          0.94220 0.23033  0.07863 0.09445  0.00000
Write Linear Equation from Situation 0.93223 0.08638  0.00955 0.11905  0.00000
Area Parallelogram                   0.91486 0.00178  0.01421 0.00000  0.00000
Circumference                        0.90694 0.10922  0.13340 0.26842  0.00000
Area Circle                          0.90594 0.15372  0.12925 0.26267  0.00000
Mode                                 0.90268 0.28140  0.04492 0.06894  0.00000
Perimeter of a Polygon               0.90091 0.00034  0.12486 0.27301  0.00000
Area Rectangle                       0.89444 0.01272  0.02427 0.02244  0.00000
Translations                         0.89113 0.02760 

In [12]:
from google.colab import files
files.download('bkt_model.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
import sys
sys.path.append('..')

import pickle
from pathlib import Path

model_path = Path('../models/bkt_model.pkl')
print(f"Model file exists: {model_path.exists()}")
print(f"Model file size: {model_path.stat().st_size / 1024:.1f} KB")

with open(model_path, 'rb') as f:
    model = pickle.load(f)

print(f"\nModel loaded successfully")
print(f"Type: {type(model).__name__}")

params = model.params()
print(f"\nNumber of skills: {params.index.get_level_values('skill').nunique()}")
print(f"\nSample parameters:")
print(params.head(10))

Model file exists: True
Model file size: 44.6 KB


ValueError: <class 'numpy.random._mt19937.MT19937'> is not a known BitGenerator module.

In [2]:
import json
from pathlib import Path

params_path = Path('../models/bkt_params.json')
print(f"File exists: {params_path.exists()}")
print(f"File size: {params_path.stat().st_size / 1024:.1f} KB")

with open(params_path) as f:
    params = json.load(f)

print(f"\nLoaded parameters for {len(params)} skills")
print(f"\nSample skills:")
for skill in list(params.keys())[:3]:
    print(f"  {skill}: {params[skill]}")

File exists: True
File size: 17.8 KB

Loaded parameters for 95 skills

Sample skills:
  Percent Of: {'prior': 0.05851251549018819, 'learns': 0.012996683982730742, 'guesses': 0.5339581704995794, 'slips': 0.0006898601772722872, 'forgets': 0.0}
  Circle Graph: {'prior': 0.5628464345577833, 'learns': 0.05845906287235211, 'guesses': 0.13951327800486416, 'slips': 0.28755331014546825, 'forgets': 0.0}
  Finding Percents: {'prior': 0.40382089864655796, 'learns': 0.08560312863684555, 'guesses': 0.11657517287282447, 'slips': 0.19921718170330052, 'forgets': 0.0}
